In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from ultralytics import YOLO

# Load YOLOv8 segmentation model
yolo_model = YOLO("yolov8n-seg.pt")


def segment_paper_yolo(img):
    """
    Returns cropped paper image using YOLOv8 segmentation
    """
    results = yolo_model(img, imgsz=640, conf=0.25, verbose=False)
    result = results[0]

    if result.masks is None:
        return None

    # Assume largest mask = paper
    masks = result.masks.data.cpu().numpy()
    areas = masks.sum(axis=(1, 2))
    paper_mask = masks[np.argmax(areas)]

    # Convert mask to uint8
    mask = (paper_mask * 255).astype(np.uint8)

    # Crop bounding box of mask
    coords = cv2.findNonZero(mask)
    x, y, w, h = cv2.boundingRect(coords)

    cropped = img[y:y + h, x:x + w]
    cropped_mask = mask[y:y + h, x:x + w]

    # Apply mask (remove background completely)
    bg = np.full_like(cropped, 255)
    cropped = np.where(cropped_mask[..., None] == 255, cropped, bg)

    return cropped


def normalize_orientation(img):
    """
    Rotate portrait paper to landscape
    """
    h, w = img.shape[:2]
    if h > w:
        img = cv2.rotate(img, cv2.ROTATE_90_CLOCKWISE)
    return img


def normalize_white_background(img):
    lab = cv2.cvtColor(img, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)

    # Flatten lighting
    l = cv2.normalize(l, None, 240, 255, cv2.NORM_MINMAX)

    lab = cv2.merge((l, a, b))
    img = cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)

    # Detect paper (low saturation + high brightness)
    hsv = cv2.cvtColor(img, cv2.COLOR_RGB2HSV)
    _, s, v = cv2.split(hsv)

    paper_mask = ((s < 20) & (v > 200)).astype(np.uint8) * 255
    kernel = np.ones((5, 5), np.uint8)
    paper_mask = cv2.morphologyEx(paper_mask, cv2.MORPH_CLOSE, kernel, iterations=2)

    img[paper_mask == 255] = [255, 255, 255]
    return img


def enhance_crayon_colors(img):
    hsv = cv2.cvtColor(img, cv2.COLOR_RGB2HSV)
    h, s, v = cv2.split(hsv)

    # Boost saturation but preserve texture
    s = np.clip(s.astype(np.float32) * 1.3 + 8, 0, 255).astype(np.uint8)
    v = np.clip(v.astype(np.float32) * 1.05, 0, 255).astype(np.uint8)

    hsv = cv2.merge((h, s, v))
    return cv2.cvtColor(hsv, cv2.COLOR_HSV2RGB)


def remove_noise_preserve_texture(img):
    return cv2.bilateralFilter(img, d=5, sigmaColor=50, sigmaSpace=50)


def crop_to_content(img):
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    mask = gray < 245
    coords = cv2.findNonZero(mask.astype(np.uint8))

    if coords is None:
        return img

    x, y, w, h = cv2.boundingRect(coords)

    pad_x = int(w * 0.05)
    pad_y = int(h * 0.05)

    x = max(0, x - pad_x)
    y = max(0, y - pad_y)
    w = min(img.shape[1] - x, w + 2 * pad_x)
    h = min(img.shape[0] - y, h + 2 * pad_y)

    return img[y:y + h, x:x + w]


def resize_to_dataset(img, target_w=512, target_h=362):
    h, w = img.shape[:2]
    scale = min(target_w / w, target_h / h)

    new_w = int(w * scale)
    new_h = int(h * scale)

    resized = cv2.resize(img, (new_w, new_h), interpolation=cv2.INTER_AREA)

    canvas = np.full((target_h, target_w, 3), 255, dtype=np.uint8)
    y_off = (target_h - new_h) // 2
    x_off = (target_w - new_w) // 2

    canvas[y_off:y_off + new_h, x_off:x_off + new_w] = resized
    return canvas


def full_pipeline(image_path):
    img = cv2.imread(image_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    paper = segment_paper_yolo(img)
    if paper is None:
        raise RuntimeError("Paper not detected")

    paper = normalize_orientation(paper)
    paper = normalize_white_background(paper)
    paper = enhance_crayon_colors(paper)
    paper = remove_noise_preserve_texture(paper)
    paper = crop_to_content(paper)
    final = resize_to_dataset(paper)

    return final


final = full_pipeline("ml-models/image/data/wood_bg1.jpeg")

plt.figure(figsize=(6, 4))
plt.imshow(final)
plt.axis("off")
plt.title("Final Dataset-Matched Image")
plt.show()

cv2.imwrite("ml-models/image/data/output/final_dataset_style.png", cv2.cvtColor(final, cv2.COLOR_RGB2BGR))


# 📌 DEBUG & VISUALIZATION CELL ()

def visualize_pipeline_steps(image_path):
    img = cv2.imread(image_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # ---- YOLO PAPER SEGMENTATION ----
    results = yolo_model(img, imgsz=640, conf=0.25, verbose=False)
    result = results[0]

    if result.masks is None:
        raise RuntimeError("Paper not detected")

    masks = result.masks.data.cpu().numpy()
    areas = masks.sum(axis=(1, 2))
    paper_mask = masks[np.argmax(areas)]

    mask_uint8 = (paper_mask * 255).astype(np.uint8)

    # Edge visualization of YOLO mask
    edges = cv2.Canny(mask_uint8, 80, 160)
    edges_rgb = np.dstack([edges] * 3)

    # ---- APPLY YOUR ORIGINAL PIPELINE STEP BY STEP ----
    paper = segment_paper_yolo(img)
    paper_oriented = normalize_orientation(paper)
    paper_white = normalize_white_background(paper_oriented)
    paper_color = enhance_crayon_colors(paper_white)
    paper_denoised = remove_noise_preserve_texture(paper_color)
    paper_cropped = crop_to_content(paper_denoised)
    final_resized = resize_to_dataset(paper_cropped)

    # ---- VISUALIZATION GRID ----
    plt.figure(figsize=(16, 10))

    plt.subplot(2, 3, 1)
    plt.imshow(img)
    plt.title("Original Input")
    plt.axis("off")

    plt.subplot(2, 3, 2)
    plt.imshow(mask_uint8, cmap="gray")
    plt.title("YOLOv8 Paper Mask")
    plt.axis("off")

    plt.subplot(2, 3, 3)
    plt.imshow(edges_rgb)
    plt.title("YOLOv8 Mask Edges")
    plt.axis("off")

    plt.subplot(2, 3, 4)
    plt.imshow(paper)
    plt.title("Paper Cropped (YOLO)")
    plt.axis("off")

    plt.subplot(2, 3, 5)
    plt.imshow(paper_cropped)
    plt.title("After Content Crop")
    plt.axis("off")

    plt.subplot(2, 3, 6)
    plt.imshow(final_resized)
    plt.title("Final (512 × 362 Dataset Match)")
    plt.axis("off")

    plt.tight_layout()
    plt.show()

    return final_resized


# ---- RUN DEBUG VISUALIZATION ----
final = visualize_pipeline_steps("ml-models/image/data/wood_bg1.jpeg")

cv2.imwrite(
    "ml-models/image/data/output/final_dataset_style.png",
    cv2.cvtColor(final, cv2.COLOR_RGB2BGR)
)